


### Retiro Fugas

In [9]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

In [10]:

filename='retiro_negocios_20260716_01.csv'
df_prueba_ppdplus=cargar_archivo_csv(spark,filename,';',True)
print(df_prueba_ppdplus.columns)  



['DNI']


In [12]:
df_prueba_ppdplus = df_prueba_ppdplus.withColumn(
    "DNI",
    F.right(
        F.concat(F.lit("00000000"), F.col("DNI")),
        F.lit(8)
    )

)

In [14]:
overwrite_table_SQL(spark,df_prueba_ppdplus,f'harto_borrado',server_zeus,user_zeus,pwd_zeus,'odin')
overwrite_table_SQL(spark,df_prueba_ppdplus,f'harto_borrado',server_sa,user_sa,pwd_sa,'odin')
overwrite_table_SQL(spark,df_prueba_ppdplus,f'harto_borrado',server_kishin,user_kishin,pwd_kishin,'DANTALION')


In [2]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from sqlalchemy import create_engine
from sqlalchemy import text


In [3]:

server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

server_sql = server_zeus
db_sql = "SAMANTHA"
user_sql = user_zeus
pwd_sql = pwd_zeus

engine_samantha = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

In [5]:
fecha_mes_base='2026-07-01'

filename='RetiroDefinitivo_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_def_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_Telefonos.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_telf = pd.read_csv(ruta_archivo,sep='|')

filename='retiro_correo_alfin.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_retiro_correo = pd.read_csv(ruta_archivo,sep=';')

df_def_blacklist = df_def_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_blacklist = df_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_telf= df_telf.rename(columns={'TELEFONO': 'celular'})
df_retiro_correo= df_retiro_correo.rename(columns={'DNI': 'dni_cliente'})


# Blacklists de DNI
df1 = df_def_blacklist.copy()
df1["celular"] = None
df1 = df1[["dni_cliente", "celular"]]

df2 = df_blacklist.copy()
df2["celular"] = None
df2 = df2[["dni_cliente", "celular"]]

# Blacklist de teléfonos
df3 = df_telf.copy()
df3["dni_cliente"] = None
df3 = df3[["dni_cliente", "celular"]]

# Archivo con DNI y celular
df4 = df_retiro_correo[["dni_cliente", "celular"]].copy()



# Unir todo
df_retiros = pd.concat(
    [df1, df2, df3, df4],
    ignore_index=True
)

dni_retiro = set(df_retiros['dni_cliente'].dropna())
cel_retiro = set(df_retiros['celular'].dropna())



C:\Users\DATA\AppData\Local\Temp\ipykernel_8256\3435753884.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_retiros = pd.concat(


In [7]:
server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
query = f"""
    select NUMERO_DOCUMENTO ,cl_telf1 as celular from DANTALION.dbo.Base_Maestra_Alfin_bk
    where FECHA_ENVIO>='2026-07-01'
"""
dfbae = pd.read_sql(query, engine_kishin)

In [8]:
dfbae = (
    ~(dfbae['NUMERO_DOCUMENTO'].isin(dni_retiro) |
    dfbae['celular'].isin(cel_retiro))
).copy()

In [9]:
filename='queda_alfin.csv'
ruta_archivo = os.path.join(ruta_csv, filename)

dfbae.to_csv(ruta_archivo,sep=';')

In [1]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from sqlalchemy import create_engine
from sqlalchemy import text

fecha_mes_base='2026-07-01'

filename='RetiroDefinitivo_BlackList.csv'
ruta_archivo = os.path.join(ruta_csv, filename)
df_def_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_BlackList.csv'
ruta_archivo = os.path.join(ruta_csv, filename)
df_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_Telefonos.csv'
ruta_archivo = os.path.join(ruta_csv, filename)
df_telf = pd.read_csv(ruta_archivo,sep='|')


print(df_def_blacklist.columns)
print(df_blacklist.columns)
print(df_telf.columns)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\DATA\\Documents\\datos\\05_subir_csv\\RetiroDefinitivo_BlackList.csv'

In [8]:
# df[f"{name_dni}"] = (
#     df[f"{name_dni}"]
#     .astype(str)
#     .str.zfill(8)
# )

# df = df.rename(columns={
#     f'{name_dni}': 'NUMERO_DOCUMENTO'
# })
# df=df[["NUMERO_DOCUMENTO"]]
# df.head()
server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
query = f"""
	select NUMERO_DOCUMENTO
	from DANTALION.dbo.Base_Maestra_Alfin_bk_Vigente
"""
df_alfin = pd.read_sql(query, engine_kishin)

In [7]:
print(df_credicash.columns.tolist())

['DNI']


In [ ]:
fecha_mes_base='2026-07-01'

filename='RetiroDefinitivo_BlackList.csv'
ruta_archivo = os.path.join(ruta_csv, filename)
df_def_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_BlackList.csv'
ruta_archivo = os.path.join(ruta_csv, filename)
df_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_Telefonos.csv'
ruta_archivo = os.path.join(ruta_csv, filename)
df_telf = pd.read_csv(ruta_archivo,sep='|')

filename='retiro_correo_alfin.csv'
ruta_archivo = os.path.join(ruta_csv, filename)
df_retiro_correo = pd.read_csv(ruta_archivo,sep=';')

df_def_blacklist = df_def_blacklist.rename(columns={'DNI': 'NUMERO_DOCUMENTO'})
df_blacklist = df_blacklist.rename(columns={'DNI': 'NUMERO_DOCUMENTO'})
df_telf= df_telf.rename(columns={'TELEFONO': 'celular'})
df_retiro_correo= df_retiro_correo.rename(columns={'DNI': 'NUMERO_DOCUMENTO'})

print(df_def_blacklist.columns)
print(df_blacklist.columns)
print(df_telf.columns)
print(df_retiro_correo.columns)
print(df_formato.columns.tolist())

In [4]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from sqlalchemy import create_engine
from sqlalchemy import text

fecha_mes_base='2026-07-01'

filename='retiro_correo_alfin.csv'
name_dni='DNI'

ruta_archivo = os.path.join(ruta_csv, filename)
df = pd.read_csv(ruta_archivo,sep=';')
df.head()

,DNI,celular,RETIRO
0,29652447,958543530,RETIRO_CORREO
1,71126728,999142834,RETIRO_CORREO
2,46121953,959333262,RETIRO_CORREO
3,8699549,988028568,RETIRO_CORREO
4,71104707,935394527,RETIRO_CORREO


In [ ]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from sqlalchemy import create_engine
from sqlalchemy import text

fecha_mes_base='2026-07-01'

filename='blacklist_celulares.txt'
name_dni='DNI'

ruta_archivo = os.path.join(ruta_csv, filename)
df = pd.read_csv(ruta_archivo,sep='|')
df.head()

,CELULAR
0,900000202
1,900000323
2,900000531
3,900000910
4,900001728


In [15]:
query = f"""
	select cl_telf1 as CELULAR
	from DANTALION.dbo.Base_Maestra_ALFCC_Vigente
    WHERE cl_estado IS not NULL OR cl_estado =1
"""
df_credicash = pd.read_sql(query, engine_kishin)

In [16]:
df["CELULAR"] = (
    df["CELULAR"]
    .astype(str)
    .str.zfill(8)
)
df_credicash["CELULAR"] = (
    df_credicash["CELULAR"]
    .astype(str)
    .str.zfill(8)
)

In [17]:
df.merge(df_credicash, on='CELULAR', how='inner').count()

CELULAR    0
dtype: int64

In [ ]:
df.merge(df_credicash, on='CELULAR', how='inner').head()

,DNI
0,22960032
1,41364472


In [ ]:

list_dni = (
    df['DNI']
    .dropna()
    .drop_duplicates()
    .tolist()
)
in_clause = ",".join(f"'{x}'" for x in list_dni)

in_clause

In [ ]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from sqlalchemy import create_engine
from sqlalchemy import text

fecha_mes_base='2026-07-01'

filename='RetiroDefinitivo_BlackList.csv'
name_dni='DNI'

ruta_archivo = os.path.join(ruta_csv, filename)
df = pd.read_csv(ruta_archivo,sep='|')
df.head()

,DNI
0,00013948
1,00015508
2,00031376
3,00035800
4,00036260


In [ ]:



df[f"{name_dni}"] = (
    df[f"{name_dni}"]
    .astype(str)
    .str.zfill(8)
)

df = df.rename(columns={
    f'{name_dni}': 'NUMERO_DOCUMENTO'
})
df=df[["NUMERO_DOCUMENTO"]]
df.head()
server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
query = f"""
	select NUMERO_DOCUMENTO
	from DANTALION.dbo.Base_Maestra_Diners_TC_Vigente
    WHERE RETIRO IS NULL OR RETIRO =''
"""
df_tc = pd.read_sql(query, engine_kishin)


In [18]:
df_tc.shape

(15332, 1)

In [6]:
campana='dinersTc'
query = f"""
    select FECHA as fecha_gestion,DNI as NUMERO_DOCUMENTO,
    PROMOTOR as promotor,
    ESTADO as estado_venta,
    TRAMA_HORA as tramo_venta,
    MONTO as monto_venta,
    DNIEjecutivo as dni_ejecutivo_venta,
    Producto as producto_venta,
    Producto as title,
    Celular as cel_venta,1 as venta,
    subcampana
    from SAMANTHA.dbo.Ventas_Target
    where campana='{campana}'
    and cast(fecha as date) between '{fecha_mes_base}' and EOMONTH('{fecha_mes_base}')
"""
df_ventas = pd.read_sql(query, engine_samantha)

In [16]:
df_ventas.count()

fecha_gestion          51
NUMERO_DOCUMENTO       51
promotor               51
estado_venta           51
tramo_venta            51
monto_venta             0
dni_ejecutivo_venta    51
producto_venta          0
title                   0
cel_venta              51
venta                  51
subcampana              0
dtype: int64

In [7]:
df = df[
    ~df["NUMERO_DOCUMENTO"].isin(df_ventas["NUMERO_DOCUMENTO"])
].copy()


In [8]:
df.merge(df_tc, on='NUMERO_DOCUMENTO', how='inner').count()


NUMERO_DOCUMENTO    0
dtype: int64

In [9]:

list_dni = (
    df['NUMERO_DOCUMENTO']
    .dropna()
    .drop_duplicates()
    .tolist()
)
in_clause = ",".join(f"'{x}'" for x in list_dni)

in_clause

"'45430231','40789975','46099939','75347511','73822062','00517831','74893815','45870423','71756439','29661693','76224741','43802335','46210758','80507642','45488385','46713995','74278103','25780353','10493240','77017172','71001134','40101852','43587345','72242431','16689437','25719415','47222574','10664118','73108124','42918633','03897961','08326458','73671498','74032468','71269972','25719677','44210437','70362663','76731883','40967222','72019934','72516586','44192315','40540834','33432547','42744395','72318000','70422399','71118755','76190783','72450169','72159782','25781210','40683623','41254338','46836584','45949570','42587313','75381780','48261107','74688040','74279262','47609634','46476351','07186103','75061201','47796338','76591509','41218453','48248727','75442136','40815447','44165132','70445682','41699113','47977001','71740655','25750955','42335385','04642001','42304877','29739186','71073282','77455849','00495438','46474205','44802668','31045146','73783524','72744592','75354783

In [24]:
in_clause = ",".join(f"'{x}'" for x in list_dni)

In [9]:
df.count()

NUMERO_DOCUMENTO    2034
dtype: int64

In [32]:
try:
    with engine_kishin.begin() as conn:
        query = f"""
            UPDATE DANTALION.dbo.Base_Maestra_Diners_TC
            SET RETIRO = 'RETIRO_01'
            WHERE NUMERO_DOCUMENTO IN ({in_clause})
                and TRY_CONVERT(DATE, fecha_envio) >= TRY_CONVERT(DATE, '{fecha_mes_base}')
                AND TRY_CONVERT(DATE, fecha_envio) < DATEADD(MONTH, 1, TRY_CONVERT(DATE, '{fecha_mes_base}'))
        """
        result = conn.execute(text(query))
        print("Filas actualizadas:", result.rowcount)

except Exception as e:
    print(e)

Filas actualizadas: 215


In [ ]:

df_ventas.head()

,fecha_gestion,NUMERO_DOCUMENTO,promotor,estado_venta,tramo_venta,monto_venta,dni_ejecutivo_venta,producto_venta,title,cel_venta,venta,subcampana
0,2026-07-01,07259135,08728081,1.-VALIDADA,9,None,08728081,None,None,981260227,1,None
1,2026-07-01,07779510,44592143,6.-SIN VALIDAR,13,None,44592143,None,None,989371280,1,None
2,2026-07-01,40102384,44592143,1.-VALIDADA,10,None,44592143,None,None,932595948,1,None
3,2026-07-01,40552162,47202133,1.-VALIDADA,10,None,47202133,None,None,988594366,1,None
4,2026-07-01,41266190,44592143,1.-VALIDADA,9,None,44592143,None,None,940049145,1,None


In [33]:
exec_query_sql(server_kishin, db_kishin, user_kishin, pwd_kishin, "tNumeros_Diners_tc", "SP tNumeros diners TC")
exec_query_sql(server_zeus, "ODIN", user_zeus, pwd_zeus, "EXEC Sp_Actualizar_Diners_tc", "SP actualizar diners TC Zeus")
exec_query_sql(server_sa, "ODIN", user_sa, pwd_sa, "EXEC Sp_Actualizar_Diners_tc", "SP actualizar diners TC SA")

SP tNumeros diners TC | realizado | duración: 192.49 seg
SP actualizar diners TC Zeus | realizado | duración: 9.92 seg
SP actualizar diners TC SA | realizado | duración: 2.35 seg


In [8]:
query = f"""
	select NUMERO_DOCUMENTO,PROB_CONTACTO,concat('2026-07-',right(retiro,2)) as retiro
	from DANTALION.dbo.Base_Maestra_Diners_TC_Vigente
    WHERE RETIRO IS not NULL OR RETIRO <>''
"""
df_tc = pd.read_sql(query, engine_kishin)

In [9]:
df_tc.head()

,NUMERO_DOCUMENTO,PROB_CONTACTO,retiro
0,47216152,D,2026-07-01
1,71561197,D,2026-07-01
2,42968074,C,2026-07-01
3,73473064,E,2026-07-01
4,46510882,C,2026-07-01


In [ ]:


server_sql = server_zeus
db_sql = "SAMANTHA"
user_sql = user_zeus
pwd_sql = pwd_zeus

engine_zeus = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
fecha_mes_base='2026-06-01'
campana=fecha_a_nombre('2026-05-01')
query = f"""
select dni as NumDoc, 1 as venta_target from SAMANTHA.dbo.Ventas_Target
where CAMPANA='Diners'
and CONVERT(DATE, FECHA) >= CONVERT(DATE, '{fecha_mes_base}')
AND CONVERT(DATE, FECHA) < DATEADD(MONTH, 1, CONVERT(DATE, '{fecha_mes_base}'))
   
"""
df_venta = pd.read_sql(query, engine_zeus)

df_target = df_tc.merge(
    df_venta,
    on='NumDoc',
    how='left'
)
df_final = df_target.merge(
    df,
    on='NumDoc',
    how='inner'
)
df_final = (
    df_final[df_final['venta_target'].isnull()]
    .drop(columns=['venta_target'])
)
df_final.rename(
    columns={
        'Importe Solicitado': 'Monto'
    },
    inplace=True
)
print(df_final.columns.tolist())

c:\Users\DATA\AppData\Local\Programs\Python\Python311\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


['NumDoc', 'TIPO_PRODUCTO', 'RETIRO', 'Canal', 'Autor', 'Subcanal', 'Motivo', 'Monto', 'fecha', 'hora']


In [72]:
df_final=df_final[df_final['Monto'].notnull()]

In [78]:
df_final[['TIPO_PRODUCTO','Canal', 'Monto', 'fecha']].head()


,TIPO_PRODUCTO,Canal,Monto,fecha
0,PPD,CANALES DIGITALES,41100.0,2026-06-04
1,PPD,CANALES DIGITALES,13800.0,2026-06-04
2,PPD,CONTACT CENTER,6000.0,2026-06-04


In [ ]:

total_monto_ppd = df_final.loc[
    df_final['TIPO_PRODUCTO'] == 'PPD',
    'Monto'
].sum()

total_ope_ppd = df_final.loc[
    df_final['TIPO_PRODUCTO'] == 'PPD',
    'Monto'
].count()

print(f'PPD | Monto total: {int(total_monto_ppd)} |  Total operaciones {int(total_ope_ppd)} ')

PPD | Monto total: 60900 |  Total operaciones 3 


In [ ]:
# ruta_archivo = os.path.join(ruta_csv, 'PPD_plus.xlsx')
# df_list_filtrada.to_excel(ruta_archivo, index=False)

In [ ]:
list_dni = (
    df_final.loc[df_final['RETIRO'].isna(), 'NumDoc']
    .dropna()
    .drop_duplicates()
    .tolist()
)

in_clause = ",".join(f"'{x}'" for x in list_dni)

try:
    with engine_kishin.begin() as conn:
        query = f"""
            UPDATE DANTALION.dbo.Base_Maestra_Diners
            SET RETIRO = 'RETIRO'
            WHERE NumDoc IN ({in_clause})
                and CONVERT(DATE, fecha_envio) >= CONVERT(DATE, '{fecha_mes_base}')
                AND CONVERT(DATE, fecha_envio) < DATEADD(MONTH, 1, CONVERT(DATE, '{fecha_mes_base}'))
        """
        result = conn.execute(text(query))
        print("Filas actualizadas:", result.rowcount)

except Exception as e:
    print(e)


In [77]:
exec_query_sql(server_kishin, db_kishin, user_kishin, pwd_kishin, "tNumeros_Diners", "SP tNumeros diners")
exec_query_sql(server_zeus, "ODIN", user_zeus, pwd_zeus, "EXEC Sp_Actualizar_Diners", "SP actualizar diners Zeus")
exec_query_sql(server_sa, "ODIN", user_sa, pwd_sa, "EXEC Sp_Actualizar_Diners", "SP actualizar diners SA")

SP tNumeros diners | realizado | duración: 5.48 seg
SP actualizar diners Zeus | realizado | duración: 5.54 seg
SP actualizar diners SA | realizado | duración: 1.5 seg


In [55]:
fecha_mes_base='2026-07-01'
tipi_cond1='diners'
tipi_cond2='plus'
# tipi_cond2='CD'
tipi_cond3='xx'
tb_tipolofia='tTipologia_Diners_PPD'
servidor_01=21
tipi_cod='cod'
tipi_resp_cod='C0'
tipi_descrip='[NIVEL 4]'
tipi_estado='[NIVEL 2]'
tipi_resp_estado='NO CONTACTO'
tipi_subdescripcion='[NIVEL 3]'
tnum_tb='tNumeroDiners'
tnum_dni='NumDoc'

tlista_generada='borrar_prestamo_diner'
get_base=since_base_maestra_pp_dinners

def resumen_vicidial(spark,fecha_mes_base,tipi_cond1,tipi_cond2,tipi_cond3,tb_tipolofia,servidor_01,tipi_cod,tipi_resp_cod,tipi_descrip,tipi_estado,tipi_resp_estado):
    query = f"""
        SELECT *
        FROM OPENQUERY([192.168.3.{servidor_01}], '
            SELECT        
            rtrim(ltrim(d.vendor_lead_code)) AS vendor_lead_code,        
            e.dial_method,
            a.campaign_id AS numero_campana,        
            a.user AS dni_ejecutivo,
            c.full_name AS ejecutivo,
            e.campaign_name AS nombre_campana,        
            a.call_date AS fecha_hora_llamada,        
            a.length_in_sec AS duracion,        
            b.status_name AS call_result,        
            f.list_description,        
            f.list_name,        
            a.phone_number as phone_number,        
            d.alt_phone as fecha_agenda,        
            d.comments as comentarios,        
            a.status AS codigo,
            a.term_reason,	
            a.alt_dial
            FROM asterisk.vicidial_log a         
            LEFT JOIN asterisk.vicidial_list d ON a.lead_id=d.lead_id        
            LEFT JOIN asterisk.vicidial_campaigns e ON a.campaign_id=e.campaign_id        
            LEFT JOIN asterisk.vicidial_lists f ON a.list_id=f.list_id        
            LEFT JOIN asterisk.vicidial_statuses b ON a.status=b.status        
            LEFT JOIN asterisk.vicidial_users c ON a.user=c.user        
            WHERE (e.campaign_name like "%{tipi_cond1}" or e.campaign_name like "%{tipi_cond2}" or e.campaign_name like "%{tipi_cond3}")
            AND a.call_date >= DATE_FORMAT(''{fecha_mes_base}'', ''%Y-%m-01'')
            AND a.call_date < 
            DATE_ADD(DATE_FORMAT(''{fecha_mes_base}'', ''%Y-%m-01''), INTERVAL 1 MONTH)
        ')

        """
    df_vicidial=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

    df_vicidial = df_vicidial.withColumn(
        "vendor_lead_code",
        F.lpad(F.col("vendor_lead_code").cast("string"), 8, "0")
    )
    query = f"""
        SELECT {tipi_cod} as codigo
        , case
            when {tipi_cod}='CALLBK' then 'VOLVER A LLAMAR - call'
            else {tipi_descrip} 
        end as descripcion
        ,case 
            when {tipi_cod}='CALLBK' then 1200
            else peso 
        end as peso 
        ,[NIVEL 2] as estado
        FROM [ODIN].[dbo].{tb_tipolofia}
        where LEFT({tipi_cod},2)='{tipi_resp_cod}' or {tipi_estado}='{tipi_resp_estado}' or {tipi_cod}='CALLBK'
        """
    df_tipi=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

    df_vicidial=df_vicidial.join(df_tipi,["codigo"],"left")

    return df_vicidial.select('fecha_hora_llamada','list_name','vendor_lead_code','phone_number','descripcion','dni_ejecutivo','ejecutivo','dial_method','term_reason','alt_dial','call_result','duracion','codigo','nombre_campana','peso','estado')

df_vici=resumen_vicidial(spark,fecha_mes_base,tipi_cond1,tipi_cond2,tipi_cond3,tb_tipolofia,servidor_01,tipi_cod,tipi_resp_cod,tipi_descrip,tipi_estado,tipi_resp_estado)
# df_vici.filter(F.col('vendor_lead_code')=='45914692').orderBy(F.col('fecha_hora_llamada').desc()).show(truncate=False)


In [64]:
query = f"""
    SELECT *,NumDoc as vendor_lead_code FROM [DANTALION].[dbo].Base_Maestra_Diners_vigente
    """
df_ssff=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)

In [57]:

window_spec = Window.partitionBy("vendor_lead_code").orderBy(F.col("peso").asc_nulls_last(),F.col("fecha_hora_llamada").desc_nulls_last())
df_vici = df_vici.withColumn("orden", row_number().over(window_spec))
df_vici=df_vici.filter(F.col('orden')==1)

In [60]:
df_vici=df_vici.withColumn('me_resultado_cet',when((F.col('estado')=='CONTACTO EFECTIVO')&(F.col('orden')==1),1).otherwise(F.lit(0)))


In [65]:
df_ssff_1=df_ssff.join(df_vici,['vendor_lead_code'],'left')

In [30]:
df_vici.select('estado').distinct().show()

+--------------------+
|              estado|
+--------------------+
|CONTACTO NO EFECTIVO|
|         NO CONTACTO|
|   CONTACTO EFECTIVO|
|                NULL|
+--------------------+



In [ ]:



df_ssff_1=df_ssff_1.withColumn('me_resultado',when(F.col('orden')==1,1).otherwise(F.lit(0)))
df_ssff_1=df_ssff_1.withColumn('me_resultado_cet',when((F.col('estado')=='CONTACTO EFECTIVO')&(F.col('me_resultado')==1),1).otherwise(F.lit(0)))


window_spec = Window.partitionBy("vendor_lead_code").orderBy(F.col("vendor_lead_code").asc_nulls_last())
df_ssff_1 = df_ssff_1.withColumn("unico", row_number().over(window_spec))
df_ssff_1=df_ssff_1.withColumn('unico',when(F.col('unico')==1,1).otherwise(F.lit(0)))


In [66]:
df_list_filtrada = df_ssff_1.toPandas()
ruta_archivo = os.path.join(ruta_csv, 'validar_base_ssff3.xlsx')
df_list_filtrada.to_excel(ruta_archivo, index=False)

In [21]:
df_vici.show()

+-------------------+---------+----------------+------------+--------------------+-------------+--------------------+--------------------+------------+--------+--------------------+--------+------+--------------+
| fecha_hora_llamada|list_name|vendor_lead_code|phone_number|         descripcion|dni_ejecutivo|           ejecutivo|         dial_method| term_reason|alt_dial|         call_result|duracion|codigo|nombre_campana|
+-------------------+---------+----------------+------------+--------------------+-------------+--------------------+--------------------+------------+--------+--------------------+--------+------+--------------+
|2026-07-01 12:00:39|      AB_|        10799771|   998667811|         NO CONTESTA|         VDAD|  Outbound Auto Dial|RATIO            ...|CALLER      |    NONE|Answering Machine...|       1|    AA|2026-07 DINERS|
|2026-07-01 12:00:40|      AB_|        07876095|   920361561|         NO CONTESTA|         VDAD|  Outbound Auto Dial|RATIO            ...|CALLER    